# 12_02.한개 페이지 크롤링(BeautifulSoup)

## 1.기본 package 설정


In [ ]:
## 1.기본
import numpy as np  # numpy 패키지 가져오기
import pandas as pd # pandas 패키지 가져오기
import matplotlib.pyplot as plt # 시각화 패키지 가져오기

## 2.크롤링
from bs4 import BeautifulSoup
import requests
from tqdm import tqdm

In [ ]:
# 네이버에서 접속 제한 풀기
# req = requests.get(news_url, headers={'User-agent': 'Mozilla/5.0'})
# soup = BeautifulSoup(req.text, "lxml")  # html에 대하여 접근할 수 있도록

In [ ]:
# 네이버에서 접속 제한 풀기
# User-Agent 확인: https://www.useragentstring.com/
header = ({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
          "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"})

## 2.검색어 설정

In [ ]:
search_url = f"https://search.naver.com/search.naver?where=news&sm=tab_jum&query=빅데이터"

## 3.URL주소 검색

In [ ]:
import requests

requests = requests.get(search_url, headers = header)
bs = BeautifulSoup(requests.text)        # header 없음
url = bs.select("div.news_wrap.api_ani_send > div > div.news_info > div.info_group > a:nth-child(3)")

news_url = []
for url in tqdm(url):
    news_url.append(url["href"])

news_url

100%|██████████| 4/4 [00:00<00:00, 30393.51it/s]


['https://n.news.naver.com/mnews/article/003/0012008608?sid=102',
 'https://n.news.naver.com/mnews/article/421/0006964221?sid=102',
 'https://n.news.naver.com/mnews/article/215/0001117029?sid=101',
 'https://n.news.naver.com/mnews/article/082/0001225130?sid=102']

## 4.뉴스 저장


In [ ]:
news_no = []
news_title = []
news_body = []
news_press = []

for i, url in tqdm(enumerate(news_url)):
    import requests                                       # 접속할때 마다 실행해야 함
    requests = requests.get(url, headers = header)            # 개별 페이지 header
    news_html = BeautifulSoup(requests.text,"html.parser")

    # 글번호
    no = i + 1
    news_no.append(no)

    # 제목
    title = news_html.select_one("#title_area > span").text
    news_title.append(title)

    # 본문
    body = news_html.select_one("#dic_area").text
    news_body.append(body)

    # 신문사
    press = news_html.select_one("img.media_end_head_top_logo_img.light_type")["title"]
    news_press.append(press)

4it [00:02,  1.65it/s]


## 5.테이블로 저장

In [ ]:
news_df = pd.DataFrame({'번호': news_no, '제목':news_title,'본문':news_body,'출판사':news_press})
news_df

,번호,제목,본문,출판사
0,1,"서울신용보증재단, 소상공인 마이데이터 서비스 시작",\n\n데이터 형태 행정서류 활용한 편리한 서류제출 지원\n\n\n\n[서울=뉴시스...,뉴시스
1,2,"강릉시, '빅데이터'로 관광트렌드 대응",\n\n강릉 관광 빅데이터 분석·실태조사 수립 사업 완료보고회당일치기 여행 증가·로...,뉴스1
2,3,"""제주도 관광 연구""…신한카드, 데이터 결합 사업 추진",\n\n\n\n\n\n신한카드는 민간 데이터전문기관으로서 가명 정보를 활용한 첫 번...,한국경제TV
3,4,"한국수산자원공단, 행안부 공공 빅데이터 분석 지원사업 선정","\n\n\n\n\n\n한국수산자원공단(이하 수산공단, 이사장 이춘우) 수산종자산업진...",부산일보


In [ ]:
news_df["본문"][0]

"\n\n데이터 형태 행정서류 활용한 편리한 서류제출 지원\n\n\n\n[서울=뉴시스]소상공인 마이데이터 서비스.(사진=서울신용보증재단 홈페이지 캡쳐) *재판매 및 DB 금지[서울=뉴시스] 권혁진 기자 = 서울신용보증재단은 이번 달부터 행정안전부 공공 마이데이터 묶음정보를 활용한 '소상공인 마이데이터 서비스(mydata.seoulshinbo.co.kr)'를 운영한다고 2일 밝혔다.공공 마이데이터는 정보 주체인 국민의 요구에 따라 행정·공공기관이 보유한 본인 행정정보를 본인 또는 제3자에게 제공하는 서비스다. 소상공인 마이데이터 서비스에서는 공공 마이데이터 기반의 정책지원금 신청 및 지원 이력 관리, 내 점포분석 보고서, 보증 고객만족도 및 정책연구를 위한 모바일 조사 등을 제공한다.재단은 '서울 소상공인 활력지원' 묶음정보를 통해 사업자등록증명서 등 최대 11종의 행정서류를 데이터 형태로 수집함에 따라 무서류, 무방문으로 사업을 지원할 수 있게 됐다.재단은 마이데이터 서비스 고도화를 통해 편리한 서류 제출에 그치지 않고 데이터를 기반으로 맞춤형 정보를 제공할 수 있는 데이터 플랫폼으로 발전시킨다는 구상이다. 지금은 소상공인 지원사업을 찾아서 여러 홈페이지를 돌아다녀야 하지만, 고도화 사업이 완료되는 내년 하반기부터는 마이데이터 동의를 통해 맞춤형 사업을 안내 받을 것으로 기대된다. 주철수 서울신용보증재단 이사장은 “생업에 바쁜 소상공인이 더 신속하고 편리하게 지원받을 수 있도록 서비스를 확대할 계획”이라며 “앞으로도 데이터 기반으로 새로운 가치를 창출해내어 서울시 소상공인을 대표하는 미래형 빅데이터 플랫폼 종합지원 전문기관으로 발전해나갈 수 있도록 노력하겠다”고 말했다．\n\n"